# Experimenting with a compiled jdsl behavior package

This notebook takes the **`retail.jdslpkg`** package that the harness compiled from live
Claude Code traces and shows how to *use* it: install jdsl from this branch, load and
inspect the package, run it deterministically on plain Python tools, and wire a model
pulled from **HuggingFace** as the backend for any residual decisions.

**What the package is.** The harness watched a frontier model repeat one task over five
customers — *look up a customer by email → list their orders → fetch the first order* —
mined the invariant structure, verified it replays, and compiled it to a portable
`.jdslpkg`. The compiled behavior needs **only `email`** as input; every id in the chain
is wired from a previous tool's result (`lookup.id → list_orders`, `list_orders.result[0].id
→ get_order`). Its residual decision burden is **0.0**, so a model is never consulted at
run time — the expensive model's behavior has been distilled into a deterministic script a
tiny model (or none) can execute.

> Run this notebook from the repository root so the relative paths resolve.

## 1. Install jdsl (this branch)

The runtime core is dependency-light; the harness/compiler live behind the `[harness]`
extra. From a clone of this branch, an **editable** install is easiest:

In [ ]:
# From the repo root (branch: harness). Editable so local package edits are picked up.
%pip install -q -e ".[harness]"

# Or install this branch straight from git (no clone needed):
# %pip install -q "jdsl[harness] @ git+https://github.com/Cantor-Industries/jdsl-py.git@harness"

In [ ]:
import jdsl, sys
print('python :', sys.version.split()[0])
print('jdsl   :', getattr(jdsl, '__version__', 'dev'))

## 2. Load and inspect the package

Loading a `.jdslpkg` verifies its manifest, file digests, and IR **before** anything runs
(it is treated like software, design §45). Inspect it to see its capabilities, the
reads/writes it will perform, and the burden metrics from compilation.

In [ ]:
from pathlib import Path
from jdsl.package import load_package

PKG = Path('examples/harness/retail.jdslpkg')
pkg = load_package(PKG)
m = pkg.manifest

print(f'{m.name} v{m.version}  [{m.format}]')
print('verification :', m.verification.get('status'),
      '| replay coverage', m.verification.get('replay_coverage'))
print('fidelity     :', m.source.get('capture_fidelity'),
      '| episodes', m.source.get('episode_count'))
print('required caps:', m.required_capabilities)
print('permissions  :', pkg.permissions())

The behavior IR is a simple sequence with the dataflow made explicit. This is what a small
model would otherwise have to *rediscover* every single run — here it is frozen:

In [ ]:
import json, zipfile
with zipfile.ZipFile(PKG) as z:
    ir = json.loads(z.read('behavior.json'))
for step in ir['root']['children']:
    args = {k: (v.get('const', v.get('ref'))) for k, v in step['arguments'].items()}
    print(f"{step['tool']:28} {args}")

## 3. Bind host tools and run it — no model needed

A package carries no implementations; you **bind** each required capability to a real
callable via a `TOOLS` dict (`logical_id → callable`). `examples/harness/retail_tools.py`
provides the four retail functions as plain in-process Python (no MCP transport needed to
*run* a compiled behavior). `as_root(TOOLS)` lowers the IR onto them, and `.run(email=...)`
executes it. Because burden is 0.0, **no model is passed** — the runtime drives the whole
chain and wires every id itself.

In [ ]:
import importlib.util

def load_tools(path):
    spec = importlib.util.spec_from_file_location('retail_tools', path)
    mod = importlib.util.module_from_spec(spec); spec.loader.exec_module(mod)
    return mod.TOOLS

TOOLS = load_tools('examples/harness/retail_tools.py')
root = pkg.as_root(TOOLS)          # bound, runnable — like any authored skill

for email in ['ada@example.com', 'bo@example.com', 'cass@example.com',
              'dev@example.com', 'el@example.com']:
    ctx = root.run(email=email)
    order = ctx.blackboard['mcp_retail_get_order_out_3']
    print(f"{email:20} -> first order {order['id']:9} status={order['status']}")

Each run consumed only `email`; the customer id and order id were **derived**, not supplied.
That is the payoff — the frontier model's judgement compiled into a deterministic program.

## 4. Where a (HuggingFace) model plugs in

When a behavior *does* keep a residual decision — a genuine semantic choice the structure
can't pin down (e.g. *which* pending order to cancel) — the runtime calls a model at that
leaf, and **only** there. The burden metric tells you how often:

| metric | this package | meaning |
|---|---|---|
| `residual_decision_burden` | **0.0** | fraction of decisions left to a model |
| `deterministic_coverage` | **1.0** | fraction the compiler made deterministic |
| `exact_dataflow_refs` | **2** | id-copies wired from prior results |

The runtime accepts **any** object exposing `.generate(system=, messages=, model_id=)` as
its model — so a HuggingFace model drops in behind a tiny adapter, no provider changes.
Here is that adapter, backed by a small instruct model from the Hub:

In [ ]:
# Optional: needs `transformers` + `torch`.  %pip install -q transformers torch
class HFResidualModel:
    """Adapts a HuggingFace chat model to jdsl's model interface. jdsl calls
    .generate(system=..., messages=[{role, content}...], model_id=...) at residual
    'predict' leaves; we render that through the model's chat template."""
    def __init__(self, model_id='Qwen/Qwen2.5-0.5B-Instruct'):
        from transformers import AutoModelForCausalLM, AutoTokenizer
        self.tok = AutoTokenizer.from_pretrained(model_id)
        self.lm = AutoModelForCausalLM.from_pretrained(model_id)

    def generate(self, *, system, messages, model_id=None):
        chat = ([{'role': 'system', 'content': system}] if system else []) + list(messages)
        ids = self.tok.apply_chat_template(chat, add_generation_prompt=True,
                                           return_tensors='pt')
        out = self.lm.generate(ids, max_new_tokens=128, do_sample=False,
                               pad_token_id=self.tok.eos_token_id)
        return self.tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True).strip()

Load it and confirm the Hub model actually answers a *residual-style* prompt (the kind of
one-line choice a leaf would pose):

In [ ]:
try:
    hf = HFResidualModel()                       # downloads from HuggingFace on first use
    reply = hf.generate(
        system='You pick exactly one order id. Answer with only the id.',
        messages=[{'role': 'user', 'content': 'Orders: O_ada_1 (shipped), O_ada_2 (pending). Which is pending?'}])
    print('HF model says:', reply)
except Exception as e:
    hf = None
    print('HF model unavailable (install transformers+torch to try):', e)

You can hand that model straight to `.run(model=hf)`. For **this** package it stays dormant
(burden 0.0 → the runtime never calls it), which you can prove — the answer is identical
with or without the model attached:

In [ ]:
ctx = root.run(email='ada@example.com', model=hf)   # model attached but never consulted
print('with HF attached ->', ctx.blackboard['mcp_retail_get_order_out_3'])
# To see the HF model actually drive a step, compile a behavior whose
# residual_decision_burden > 0 (a real semantic choice) — then that leaf routes to `hf`.

## 5. Things to experiment with

- **Vary the input.** `root.run(email=...)` over your own emails; watch ids get derived.
- **Swap tool implementations.** Point `TOOLS` at a real store/API (same logical ids, same
  return *shapes*) and the identical package now drives production data — the behavior is
  portable, the bindings are yours.
- **Inspect intermediate state.** The whole `ctx.blackboard` is the run's dataflow trace.
- **Break a wire on purpose** to see the runtime's path-extraction fail loudly (e.g. make
  `list_orders` return a bare list instead of `{"result": [...]}`).
- **Swap the Hub model** in `HFResidualModel(model_id=...)` for any chat model you like.

In [ ]:
# Unknown email -> the bound tool raises, surfaced by the run (no silent wrong answer):
try:
    root.run(email='nobody@example.com')
except Exception as e:
    print('expected failure:', type(e).__name__, '-', e)